In [1]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np



In [2]:
class ScaledDotProductAttention(layers.Layer):
    def __init__(self, d_model):
      super(ScaledDotProductAttention, self).__init__()
      self.d_model = d_model

    def call(self, query, key, value, mask = None):
      matmul_qk = tf.matmul(query, key, transpose_b = True)

      scaled_attention_logits = matmul_qk / tf.math.sqrt(tf.cast(self.d_model, tf.float32))

      if mask is not None:
        mask = tf.expand_dims(mask, 1)
        mask = tf.expand_dims(mask, 2)
        scaled_attention_logits += (mask * -1e9)

      attention_weights = tf.nn.softmax(scaled_attention_logits, axis = -1)

      output = tf.matmul(attention_weights, value)

      return output, attention_weights


In [6]:
# Multi Head Attention

class MultiHeadAttention(layers.Layer):
  def __init__(self, d_model, num_heads):
    super(MultiHeadAttention, self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model

    assert d_model % self.num_heads == 0

    self.depth = d_model // self.num_heads

    self.query_dense = layers.Dense( d_model)
    self.key_dense = layers.Dense( d_model)
    self.value_dense = layers.Dense(d_model)

    self.attention = ScaledDotProductAttention(d_model)

    self.dense_output = layers.Dense(d_model)

  def call(self, query, key, value, mask = None):
    batch_size = tf.shape(query)[0]

    query = self.query_dense(query)
    key = self.key_dense(key)
    value = self.value_dense(value)

    query = self.split_into_heads(query, batch_size)
    key = self.split_into_heads(key, batch_size)
    value = self.split_into_heads(value, batch_size)

    attention_output, attention_weights = self.attention(query, key, value, mask)

    attention_output = tf.self.combine_heads(attention_output, batch_size)
    output = self.dense_output(attention_output)


    return output, attention_weights

  def split_into_heads(self, x, batch_size):
    x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
    return tf.transpose(x, perm = [0, 2, 1, 3])


  def combine_heads(self, x, batch_size):
    x = tf.transpose(x, perm = [0, 2, 1, 3])
    return tf.reshape(x, (batch_size, -1, self.d_model))

In [7]:
class FeedForwardNetwork(layers.Layer):
  def __init__(self, d_model, d_ff):
    super(FeedForwardNetwork, self).__init__()
    self.dense1 = layers.Dense(d_ff, activation = "relu")
    self.dense2 = layers.Dense(d_model)

  def call(self, x):
    x= self.dense1(x)
    return self.dense2(x)

In [8]:
# Encoder layer

class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff):
        super(EncoderLayer, self).__init__()

        self.multi_head_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward_network = FeedForwardNetwork(d_model, d_ff)
        self.layer_norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layer_norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(0.1)
        self.dropout2 = layers.Dropout(0.1)

    def call(self,x,mask=None):
      # Multi- head attention
      attn_output, _ = self.multi_head_attention(x,x,x,mask)
      attn_output= self.dropout1(attn_output)
      out1 = self.layer_norm1(x + attn_output)

      # Feed forward network
      ffn_output = self.feed_forward_network(out1)
      ffn_output = self.dropout2(ffn_output)
      return self.layer_norm2(out1 + ffn_output)

In [9]:
class DecodeLayer(layers.Layer):
  def __init__(self, d_model, num_heads, d_ff, ):
    super(DecodeLayer, self).__init__()
    self.multi_head_attention1 = MultiHeadAttention(num_heads, d_model)
    self.multi_head_attention2 = MultiHeadAttention(num_heads, d_model)
    self.feed_forward = FeedForwardNetwork(d_model, d_ff)
    self.layernorm1 = layers.LayerNormalization(epsilon = 1e-6)
    self.layernorm2 = layers.LayerNormalization(epsilon = 1e-6)
    self.layernorm3 = layers.LayerNormalization(epsilon = 1e-6)
    self.dropout1 = layers.Dropout(0.1)
    self.dropout2 = layers.Dropout(0.1)
    self.dropout3 = layers.Dropout(0.1)

  def call(self, x, enc_output, look_ahead_mask, padding_mask):

    # Multi-head attention (self attention)
    attn1, _ = self.multi_head_attention1(x, x, x, look_ahead_mask)
    attn1 = self.dropout1(attn1)
    out1 = self.layer_norm1(attn1  + x)

    # multi-head attention (encoder- decoder attention )

    attn2,_ = self.multi_head_attention2(out1,enc_output,enc_output,padding_mask)
    attn2 = self.dropout2(attn2)
    out2 = self.layer_norm2(attn2 + out1)

    # Feed Forward network
    ff_output = self.feed_forward(out2)
    ff_output = self.dropout3(ff_output)
    return self.layer_norm3(ff_output + out2)


In [10]:
class Transformer(tf.keras.Model):
  def __init__(self, num_heads, d_model, num_layers, d_ff, vocab_size, max_len):
    super(Transformer, self).__init__()
    self.d_model = d_model
    self.num_layers = num_layers

    self.embedding = layers.Embedding(vocab_size, d_model)
    self.positional_embedding = self.get_positional_encoding(max_len, d_model)

    self.encoders = [EncoderLayer(num_heads, d_model, d_ff) for _ in range(num_layers)]
    self.decoders = [DecoderLayer(num_heads, d_model,  d_ff) for _ in range(num_layers)]

    self.final_dense = layers.Dense(vocab_size)

  def get_positional_encoding(self, max_len, d_model):
    pos = np.arrange(max_len)[:, np.newaxis]
    i = np.arrange(d_model)[np.newaxis, :]
    angles = pos / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    pos_encoding = np.concatenate([np.sin(angles[:, 0::2]), np.cos(angles[:,1::2])], axis=-1)
    return tf.cast(pos_encoding, dtype = tf.float32)

  def call(self, x, enc_output, look_ahead_mask= None, padding_mask=None):
    x = self.embedding(x) + self.positional_embedding[:tf.shape(x)[1], :]
    for i in range(self.num_layers):
      x = self.encoders[i](x, padding_mask)

    for i in range(self.num_layers):
      x= self.decoders[i](x, enc_output, look_ahead_mask, padding_mask)

    return self.final_dense(x)

# Tasks

In [12]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [13]:
result = classifier("I am happy to learn NLP")
print(result)

[{'label': 'POSITIVE', 'score': 0.9998495578765869}]


In [14]:
result = classifier("I do not like it")
print(result)

[{'label': 'NEGATIVE', 'score': 0.998218834400177}]


In [2]:
from transformers import pipeline

generator = pipeline("text-generation")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [3]:
result = generator("I am happy to learn NLP")
print(result)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': "I am happy to learn NLP and work with you to develop a comprehensive plan tailored to your needs. I can provide you with the following:\n\n1. **NLP Fundamentals**: I'll start with the basics of NLP, covering the core techniques, principles, and concepts that form the foundation of the field.\n\n2. **Practical Applications**: I'll guide you through hands-on exercises and practice sessions to help you apply NLP techniques in real-world situations, such as communication, conflict resolution, and personal development.\n\n3. **Modeling and Anchoring**: We'll explore the art of modeling, where you'll learn to identify and replicate successful behaviors, and anchoring, where you'll learn to create and use powerful anchors to influence states and behaviors.\n\n4. **Meta-Programming**: This is the study of how people structure their thoughts and perceptions. We'll delve into the world of meta-programming, where you'll learn to understand and influence the underlying beliefs

In [4]:
result = generator("Once upon a time, there was a knight")
print(result)

[{'generated_text': 'Once upon a time, there was a knight who lived in a kingdom far, far away. He was known for his bravery and chivalry, but also for his love of good food and wine. One day, he decided to go on a journey to find the most delicious dish from all the kingdoms in the land.\n\nAs he traveled from kingdom to kingdom, he tasted all sorts of amazing foods. He tried spicy stews in the southern lands, sweet pastries in the eastern lands, and rich cheeses in the western lands. But nothing could compare to the flavors he discovered in the land of Saffron.\n\nIn Saffron, he found a dish that was beyond his wildest dreams – a golden chicken soup with a secret ingredient that he had never tasted before. The soup was so delicious that it made his taste buds dance with joy. The knight decided to take a taste of the secret ingredient, and it was like nothing he had ever experienced before. It was a flavor that was both familiar and yet, utterly new.\n\nAs he tasted the secret ingredi